In [ ]:
!uv add pillow

Resolved 40 packages in 2.02s
Prepared 1 package in 5.74s
Installed 1 package in 143ms
 + pillow==12.2.0


# Dataset Processing
1. Map the dhanmondi data with physician table for more information about the doctors.
these columns are taken for next process
```python
['PRSID', 'PHYID', 'PHYNM', 'PHYDEGR', 'PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA', 'INS_DST', 'CH_ADD', 'CH_DST', 'CH_THA', 'PHY_GEND', 'BMDC_REGNO', 'PHYNM_DT', 'CHNM_DT', 'IMAGE_PATH']
```
2. Pre-process images for fintuning (gray scale convertion, height,width ratio solve, contrast increase)

3. Prepare the image `ground truth` using mapped data.

## 1. Dataset mapping with physician table

In [21]:
import pandas as pd
import os
import re

def extract_pr(path):
    if not isinstance(path, str):
        return None
    # Extract the filename part starting with PR (case insensitive just in case, but usually uppercase)
    match = re.search(r'(PR[A-Z0-9_]+)', os.path.basename(path), re.IGNORECASE)
    if match:
        return match.group(1)
    return None
def process_data(output_path):
    # Paths
    csv_path = "data/doctor_image_details (1).csv"
    excel_path = "data/R232C6_Dhanmondi_Data (2).xlsx"
    output_path = f"{output_path}/mapped_doctor_data.csv"

    print(f"Loading CSV: {csv_path}")
    df_csv = pd.read_csv(csv_path)
    print(f"CSV loaded. Sample data:\n{df_csv.head(2)}")

    # Extract PR from path
    # User says: "starting with pr is the pr"
    # Example: /content/drive/MyDrive/All_Image/PR232C6DHK23_P001.jpg -> PR232C6DHK23_P001
    

    df_csv['extracted_pr'] = df_csv['IMAGE_NAME_WITH_PATH'].apply(extract_pr)
    print(f"Extracted PR sample: {df_csv['extracted_pr'].head().tolist()}")

    print(f"Loading Excel: {excel_path}")
    # Load Excel - using the first sheet by default as checked before
    df_xl = pd.read_excel(excel_path)
    print(f"Excel loaded. Sample data:\n{df_xl.head(2)}")
    
    # Let's check for case insensitive match too
    df_xl['IMG_NM_upper'] = df_xl['IMG_NM'].astype(str).str.upper()
    df_csv['extracted_pr_upper'] = df_csv['extracted_pr'].astype(str).str.upper()

    print("Merging with Dhanmondi Data...")
    # Join on IMG_NM as it's the actual link between the files
    merged_df = pd.merge(
        df_csv, 
        df_xl, 
        left_on='extracted_pr_upper', 
        right_on='IMG_NM_upper', 
        how='left'
    )

    # Now load Physicianlist
    physician_list_path = "data/Physicianlist.xlsx"
    print(f"Loading Physician List: {physician_list_path}")
    df_physician = pd.read_excel(physician_list_path)
    
    # In Dhanmondi data it is 'PHYID', in Physicianlist it is 'PHY_ID'
    print("Merging with Physician List...")
    merged_df = pd.merge(
        merged_df,
        df_physician,
        left_on='PHYID',
        right_on='PHY_ID',
        how='left',
        suffixes=('', '_phy')
    )
    merged_df["IMAGE_PATH"] = merged_df['IMAGE_NAME_WITH_PATH'].str.split('/').str[-1]  # Keep only the filename for clarity
    # Clean up temporary columns and unwanted columns
    columns_to_drop = [
        'IMAGE_NAME_WITH_PATH','extracted_pr', 'extracted_pr_upper', 'IMG_NM_upper',
        'DOCTOR_DETAILS_COMBINED', 'MONTH', 'ROUND', 'YEAR', 'BOOKID', 'SHOPID', 
        'CDATE', 'PDATE', 'PRSTYPE', 'PSCSLNO', 'PHY_ID', 'PHY_NM', 'PHY_DEG',
        'VC2', 'NAME', 'GP', 'QTPRS', 'QTPURCH', 'CYCLE', 'FICODE', 'OPERATOR', 
        'DIAGCD', 'DIAGNAME', 'DIAGOPTR', 'DIAGEDTR', 'GENDER', 'AGE', 'PHYSPCD', 
        'CINSTCD', 'EDATE', 'ETIME', 'DIAEDATE', 'DIAETIME', 'SCHDSLT', 'FSCODE', 
        'EDITOR', 'EDDATE', 'ROUND_phy', 'CINSTCD_phy', 'MCODE', 'MARKET', 
        'PHYSP_C', 'FICODE_phy', 'SC', 'NOTE', 'PD03', 'PD04', 'DUPLICATE', 
        'OLDCODE', 'SHEETNO', 'EDITDATE', 'EDITOR_phy', 'UNICODE', 'CYCLE_phy', 
        'DSDCODE', 'MCHCODE', 'CH_PHNO1', 'CH_PHNO2', 'CH_PHNO3', 'PHY_PHNO', 
        'PHYEMAIL', 'PHYNM_DT_DUP', 'PHYNM_ALL_DUP', 'CHNM_DT_DUP', 'CHNM_ALL_DUP','IMG_NM','DIVISION','HINSTCD','OPERATO','REGION','DT'
    ]
    
    # Filter columns_to_drop to only those that exist in the dataframe
    existing_drops = [c for c in columns_to_drop if c in merged_df.columns]
    merged_df = merged_df.drop(columns=existing_drops)

    print(f"Merge complete. Rows in CSV: {len(df_csv)}, Rows in Merged: {len(merged_df)}")
    print(f"Matched rows in Dhanmondi: {merged_df['PRSID'].notna().sum() if 'PRSID' in merged_df.columns else 'N/A'}")
    print(f"Remaining columns: {merged_df.columns.tolist()}")

    # Save to CSV
    merged_df.to_csv(output_path, index=False)
    print(f"Result saved to: {output_path}")


In [22]:
source_dir = "data"
process_data(source_dir)

Loading CSV: data/doctor_image_details (1).csv
CSV loaded. Sample data:
                                IMAGE_NAME_WITH_PATH  \
0  /content/drive/MyDrive/All_Image/PR232C6DHK23_...   
1  /content/drive/MyDrive/All_Image/PR232C6DHK23_...   

                          DOCTOR_DETAILS_COMBINED  
0  DR S M SIDDIQUR RAHMAN, MBBS, D-CARD, MD, FACC  
1                    DR MD. NUR HOSSAIN, MBBS, MD  
Extracted PR sample: ['PR232C6DHK23_P001', 'PR232C6DHK23_P002', 'PR232C6DHK23_P003', 'PR232C6DHK23_P004', 'PR232C6DHK23_P005']
Loading Excel: data/R232C6_Dhanmondi_Data (2).xlsx
Excel loaded. Sample data:
            PRSID  MONTH  ROUND  YEAR  BOOKID   SHOPID      CDATE      PDATE  \
0  PRS232C6001763    232    232  2026  DHK648  DHK1439 2026-04-11 2026-04-05   
1  PRS232C6001763    232    232  2026  DHK648  DHK1439 2026-04-11 2026-04-05   

   PRSTYPE  PSCSLNO  ... CINSTCD      EDATE     ETIME DIAEDATE DIAETIME  \
0        1       38  ...   H1164 2026-04-11  22:09:50      NaT      NaN   
1      

## 2. Image Processing

In [28]:
from PIL import ImageEnhance
from PIL import Image
import pandas as pd
import os

def process_image(image,max_width):
    """
    1. convert to gray scale
    2. resize to max_width while maintaining aspect ratio
    3. increase contrast
    """
    image = image.convert('L')  # Convert to grayscale
    
    if image.width > max_width:
        aspect_ratio = image.height / image.width
        new_height = int(max_width * aspect_ratio)
        image = image.resize((max_width, new_height))  # Resize while maintaining aspect ratio

    # Increase contrast
    image_enhanced = ImageEnhance.Contrast(image)
    image_enhanced = image_enhanced.enhance(2)  # Adjust the contrast level (2 is an example)
    return image_enhanced
    
def preprocess_images(source_dir, output_dir,max_width=512):
    image_paths = [os.path.join(source_dir, f) for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_paths = image_paths[:10]  # Process only the first 10 images
    for image_path in image_paths:
        image = Image.open(image_path)
        processed_image = process_image(image, max_width=max_width)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, os.path.basename(image_path))
        processed_image.save(output_path,format='JPEG', quality=85, optimize=True)        

In [29]:
source_directory = 'All_Image'
output_directory = 'Processed_images'
max_width = 512
preprocess_images(source_directory, output_directory)

## 3. Ground truth prepare

In [ ]:
import pandas as pd
